In [3]:
#only for colab
!git clone -qqq https://github.com/tpitois/FLBO
%cd FLBO
!pip install -q libigl pyvista pyvista[jupyter]

/content/FLBO/FLBO
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 6.7 MB/s eta 0:00:00


In [6]:
import igl
import scipy.sparse.linalg as sla
import numpy as np
import scipy.sparse as sp
from pathlib import Path
from src.geometry.finsler import Options
from src.utils.viz import plot_mesh, plot_diffusions_comparison
from src.utils.notebook import saddle_mesh
import pyvista as pv

pv.global_theme.notebook = True
pv.set_jupyter_backend('html')

# Heat Diffusion using the FLBO

## 1. Mesh Loading and Preprocessing
We begin by loading a 3D mesh (in this case, the classic Stanford Bunny). To ensure reasonable computation times for the spectral decomposition and diffusion steps while preserving the core geometry, we decimate the mesh to a lower resolution.

In [7]:
!wget -q -O bunny.off https://raw.githubusercontent.com/pmp-library/pmp-library/main/data/off/bunny.off

In [8]:
mesh_path = Path('bunny.off')
V, F = igl.read_triangle_mesh(mesh_path)

In [9]:
V, F, J, I = igl.decimate(V, F, 5000)

In [10]:
plotter = plot_mesh(V, F)
plotter.show()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

## 2. Extracting Local Frames and Principal Curvatures

To construct the Finsler-Laplace-Beltrami Operator (FLBO), we first need to define an anisotropic metric on the surface. Unlike standard isotropic diffusion, where heat spreads equally in all directions, the FLBO allows heat to diffuse preferentially along specific geometric directions.

To achieve this, we build a local coordinate system for every triangle in the mesh based on its curvature. Following the discretization scheme, for each triangle $ijk \in F$, we define an orthonormal reference frame $U_{ijk}$:

$$U_{ijk} = (\hat{u}_{M}, \hat{u}_{m}, \hat{n})$$

Where:
* $\hat{n} \in \mathbb{R}^3$ is the unit normal vector of the face.
* $\hat{u}_{M} \in \mathbb{R}^3$ is the direction of maximum principal curvature.
* $\hat{u}_{m} \in \mathbb{R}^3$ is the direction of minimum principal curvature.

In [11]:
def compute_local_frames(V, F):
    n = igl.per_face_normals(V, F, np.array([0.0, 0.0, 1.0]))

    pd1, pd2, pv1, pv2, bad_vertices = igl.principal_curvature(V, F)

    num_faces = F.shape[0]

    U = np.zeros((num_faces, 3, 3))

    for f_idx in range(num_faces):
        face = F[f_idx]
        n_f = n[f_idx]

        dir_max = pd1[face[0]]

        u_M_f = dir_max - np.dot(dir_max, n_f) * n_f

        norm = np.linalg.norm(u_M_f)
        if norm > 1e-8:
            u_M_f = u_M_f / norm
        else:
            edge = V[face[1]] - V[face[0]]
            u_M_f = edge - np.dot(edge, n_f) * n_f
            u_M_f = u_M_f / np.linalg.norm(u_M_f)

        u_m_f = np.cross(n_f, u_M_f)

        U[f_idx, :, :] = [u_M_f, u_m_f, n_f]

    return U

## 3. Computing the Finsler Diffusivity Tensor ($D_{\mathcal{F}^*}$)

This step constructs the per-face diffusivity tensor derived from a Finsler metric. This tensor defines the anisotropic and asymmetric propagation of the heat flow. The construction follows a specific geometric pipeline:

**1. Local Frame Rotation**
We construct a transformation matrix $R(\theta)$ using a user-defined orientation angle $\theta$. The local frame $U$ is rotated to align with the desired principal directions of anisotropy:
$$\tilde{U} = R(\theta) U$$

**2. Base Metric Tensor ($H$)**
An anisotropic scaling matrix $D = \text{diag}(\frac{1}{1+\alpha}, 1, 1)$ (controlled by the parameter $\alpha$) modulates the rotated frame to define the base metric tensor $H$:
$$H = \tilde{U}^\top D \tilde{U}$$

**3. Drift Vector ($\mathbf{w}$)**
To introduce asymmetry (directed flow), we extract a drift vector from the second row of the transformed frame ($\tilde{\mathbf{u}}_2$) and scale it by the drift factor $\tau$:
$$\mathbf{w} = \tau \tilde{\mathbf{u}}_2$$

**4. Dual Metric Components**
Using the scaling factor $\eta = 1 - \tau^2$, we compute the dual drift vector $\mathbf{w}^*$ and the dual metric matrix $M^*$:
$$\mathbf{w}^* = -\frac{1}{\eta} H\mathbf{w}$$
$$M^* = \frac{\eta H + (H\mathbf{w})(H\mathbf{w})^\top}{\eta^2}$$

**5. Final Diffusivity Tensor**
The final Finsler diffusivity tensor $D_{\mathcal{F}^*}$ is computed from the dual components. We apply a thresholding step to eliminate numerical noise near zero and one:
$$D_{\mathcal{F}^*} = M^* - \mathbf{w}^* (\mathbf{w}^*)^\top$$

In [12]:
def compute_D_finsler(U: np.ndarray, options: Options) -> np.ndarray:
    c = np.cos(options.angle)
    s = np.sin(options.angle)
    R = np.array([[s, c, 0], [c, -s, 0], [0, 0, 1]])

    # (N_faces,3,3)
    U_rot = R @ U

    D_mat = np.diag([1.0 / (1.0 + options.alpha), 1.0, 1.0])

    # (N_faces,3,3)
    H = U_rot.transpose(0, 2, 1) @ D_mat @ U_rot

    eta = 1.0 - (options.tau ** 2)

    # (N_faces, 3)
    U2 = U_rot[:, 1, :]
    w = options.tau * U2

    # H @ w -> (N_faces,3)
    Hw = np.einsum("mij,mj->mi", H, w)
    wstar = -Hw / eta

    # (N_faces,3,3)
    Mstar = (eta * H + np.einsum("mi,mj->mij", Hw, Hw)) / (eta ** 2)

    # (N_faces,3,3)
    D_finsler = Mstar - np.einsum("mi,mj->mij", wstar, wstar)

    D_finsler[np.abs(D_finsler) < 1e-10] = 0.0
    D_finsler[np.abs(D_finsler - 1.0) < 1e-10] = 1.0

    return D_finsler

## 4. Assembling the Anisotropic Stiffness Matrix ($W_{FLBO}$)

This step builds the sparse anisotropic stiffness matrix. It generalizes the standard cotangent Laplacian by modulating the local mesh geometry with our per-face anisotropy tensor $D_{\mathcal{F}^*}$.

**1. Interior Angles**
For each triangle with vertices $v_i, v_j, v_k$, we compute the interior angles. For example, the angle $\alpha_i$ at $v_i$ is derived from the edge vectors $\mathbf{p} = v_j - v_i$ and $\mathbf{q} = v_k - v_i$:
$$\alpha_i = \arccos(\hat{\mathbf{p}} \cdot \hat{\mathbf{q}})$$

**2. Anisotropic Edge Projections**
The unit edge vectors $\hat{\mathbf{e}}_1 = \frac{v_k - v_j}{\|v_k - v_j\|}$ and $\hat{\mathbf{e}}_2 = \frac{v_i - v_k}{\|v_i - v_k\|}$ are evaluated against the diffusivity tensor $D_{\mathcal{F}^*}$ to capture the directional weights:
$$w_{12} = \hat{\mathbf{e}}_1^\top D_{\mathcal{F}^*} \hat{\mathbf{e}}_2, \quad w_{11} = \hat{\mathbf{e}}_1^\top D_{\mathcal{F}^*} \hat{\mathbf{e}}_1$$

**3. Generalized Cotangent Weights**
The classical Laplacian weights are modified to account for the anisotropy:
* **Off-diagonal ($W_{ij}$):** Coupling weight between adjacent vertices $v_i$ and $v_j$:
  $$W_{ij} = -\frac{1}{2} \frac{w_{12}}{\sin(\alpha_k)}$$
* **Diagonal ($W_{ii}$):** Self-weight for vertex $v_i$:
  $$W_{ii} = -\frac{1}{2} w_{11} (\cot(\alpha_j) + \cot(\alpha_k))$$

**4. Sparse Assembly**
Finally, these local per-triangle weights are mapped to their global vertex indices to assemble the symmetric sparse CSR matrix $W_{FLBO}$.

In [13]:
def compute_stiffness_matrix(V, F, D_F):
    n_vertices = V.shape[0]

    angles = np.zeros_like(F, dtype=float)
    for i in range(3):
        i1, i2, i3 = i, (i + 1) % 3, (i + 2) % 3

        # (N_VERTICES, 3)
        pp = V[F[:, i2]] - V[F[:, i1]]
        qq = V[F[:, i3]] - V[F[:, i1]]

        norm_pp = np.maximum(np.linalg.norm(pp, axis=1, keepdims=True), 1e-12)
        norm_qq = np.maximum(np.linalg.norm(qq, axis=1, keepdims=True), 1e-12)
        pp /= norm_pp
        qq /= norm_qq

        # (N_VERTICES)
        dot_prod = np.sum(pp * qq, axis=1)
        dot_prod = np.clip(dot_prod, -1.0, 1.0)
        angles[:, i1] = np.arccos(dot_prod)

    I_W, J_W, V_W = [], [], []
    for i in range(3):
        i1, i2, i3 = i, (i + 1) % 3, (i + 2) % 3

        # (N_VERTICES, 3)
        e1 = V[F[:, i3]] - V[F[:, i2]]
        e2 = V[F[:, i1]] - V[F[:, i3]]

        norm_e1 = np.maximum(np.linalg.norm(e1, axis=1, keepdims=True), 1e-12)
        norm_e2 = np.maximum(np.linalg.norm(e2, axis=1, keepdims=True), 1e-12)
        e1 /= norm_e1
        e2 /= norm_e2

        # (N_VERTICES)
        e1_D_e2 = np.einsum("mi,mij,mj->m", e1, D_F, e2)
        e1_D_e1 = np.einsum("mi,mij,mj->m", e1, D_F, e1)

        # (N_VERTICES)
        sin_i3 = np.sin(angles[:, i3])
        cot_i2 = 1.0 / np.tan(angles[:, i2])
        cot_i3 = 1.0 / np.tan(angles[:, i3])

        # (N_VERTICES)
        factore = -0.5 * e1_D_e2 / sin_i3
        factord = -0.5 * e1_D_e1 * (cot_i2 + cot_i3)

        I_W.append(F[:, i1])
        J_W.append(F[:, i2])
        V_W.append(factore)

        I_W.append(F[:, i2])
        J_W.append(F[:, i1])
        V_W.append(factore)

        I_W.append(F[:, i1])
        J_W.append(F[:, i1])
        V_W.append(factord)

    I_W = np.concatenate(I_W)
    J_W = np.concatenate(J_W)
    V_W = np.concatenate(V_W)

    W = sp.coo_matrix((V_W, (I_W, J_W)), shape=(n_vertices, n_vertices)).tocsr()

    return W

In [14]:
def compute_mass_matrix(V: np.ndarray, F: np.ndarray) -> sp.csr_matrix:
    n_vertices = V.shape[0]

    v1, v2, v3 = V[F].transpose(1, 0, 2)

    triangle_areas = 0.5 * np.linalg.norm(np.cross(v2 - v1, v3 - v1), axis=1)

    vertex_masses = np.bincount(
        F.flatten(), weights=np.repeat(triangle_areas / 3.0, 3), minlength=n_vertices
    )

    return sp.diags(vertex_masses, format="csr")

## 5. Direct Heat Diffusion Simulation (Spatial Domain)

We can physically observe the FLBO behavior by simulating heat diffusion directly on the manifold. The heat equation is governed by:

$$\frac{\partial f}{\partial t} = -\Delta_{FLBO} f = -S_{FLBO}^{-1} W_{FLBO} f$$

To simulate this numerically over time, we use the backward Euler implicit scheme. Given an initial heat distribution $f_0$ (e.g., a Dirac delta at a specific vertex) and a time step $t$, we solve the following sparse linear system for the heat distribution $f_t$ at time $t$:

$$(S_{FLBO} + t W_{FLBO}) f_t = S_{FLBO} f_0$$

By varying the anisotropy parameter $\alpha$ and the orientation angle $\theta$, we can visualize how the Finsler metric actively steers the heat flow, breaking the symmetric diffusion seen in classical Riemannian geometry.

In [15]:
def solve_heat_diffusion(V, F, source, t, options, U=None, source_value=10.0):
    if U is None:
        U = compute_local_frames(V, F)
    D_F = compute_D_finsler(U, options)
    W = compute_stiffness_matrix(V, F, D_F)
    S = compute_mass_matrix(V, F)

    f0 = np.zeros(V.shape[0])
    f0[source] = source_value

    return sla.spsolve(S - t * W, S @ f0)

In [16]:
source = 546  # source for bunny

In [17]:
ft = solve_heat_diffusion(V, F, source, 1, options=Options(alpha=0, angle=0, tau=0.1))

Here, we compare the heat propagation using different parameters $(\alpha, \theta)$ to demonstrate the effect of the local frames.

In [18]:
plotter = plot_diffusions_comparison(
    V, F, source, 1, solve_heat_diffusion,
    options_list=[
        Options(alpha=0, angle=0, tau=0.1),
        Options(alpha=10, angle=0, tau=0),
        Options(alpha=10, angle=np.pi/4, tau=0),
    ]
)
plotter.camera_position = [
    (0.1703412102305781, 0.4528868362215103, -0.1069106976379307),
    (-0.016792558421875, 0.10945231280937501, -0.0013474376156249988),
    (0.12152971381020594, 0.2305337400507031, 0.9654453497528714)
]
plotter.show()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

**Observations:**
* **Left ($\alpha=0$):** We observe classical isotropic heat diffusion. The heat spreads uniformly in all directions, forming circular isolines.
* **Center ($\alpha=10, \theta=0$):** Anisotropy is introduced. The heat propagates significantly faster along the direction of maximum principal curvature, stretching the heat signature.
* **Right ($\alpha=10, \theta=\frac{\pi}{4}$):** The anisotropic diffusion is rotated by $45^\circ$. This demonstrates how the heat spread can be explicitly aligned with an arbitrarily rotated frame on the tangent plane.

In [19]:
V_saddle, F_saddle, saddle_point = saddle_mesh(1000)

In [20]:
plotter = plot_diffusions_comparison(
    V_saddle, F_saddle, saddle_point, 10, solve_heat_diffusion,
    options_list=[
        Options(alpha=0, angle=0, tau=0.1),
        Options(alpha=50, angle=0, tau=0.1),
        Options(alpha=50, angle=0, tau=0.5),
    ]
)
plotter.camera_position = [
    (1.2412249319367663, 3.214095466819236, 5.9082970293880495),
    (0.0, 0.0, 0.0),
    (-0.5692943181175201, -0.6660647049627499, 0.48193546057744385)
]
plotter.show()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

**The effect of the parameter $\tau$:**
These visualizations on a saddle mesh highlight the specific impact of the asymmetric Finsler parameter $\tau$. As $\tau$ approaches its maximum limit (close to 1), it induces a strong "drift" or "wind" effect. The heat is not just stretched; it is actively pushed forward along the target direction, demonstrating the asymmetric nature of the Randers metric.